In [7]:
import pandas as pd
import os
import re
import glob

In [8]:
folder_path = "/Users/edwin2/Desktop/DM projektas/nusikalstamumas"

# Surandame visus CSV failus
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

for file in csv_files:

    filename = os.path.basename(file)

    # Ištraukiame metus (4 skaitmenys)
    match = re.search(r"\d{4}", filename)

    if match:
        year = match.group()

        # Sukuriame DataFrame
        df = pd.read_csv(file, sep=';')

        # Sukuriame kintamojo pavadinimą
        variable_name = f"nusikalstamumas_{year}"

        # Išsaugome kaip atskirą DataFrame kintamąjį
        globals()[variable_name] = df


df_list = []

for var_name in list(globals().keys()):
    if re.match(r"nusikalstamumas_\d{4}", var_name):
        df = globals()[var_name].copy()
        year = int(var_name.split("_")[1])
        df["Metai"] = year
        df_list.append(df)

nusikalstamumas_all = pd.concat(df_list, ignore_index=True)

In [9]:
nusikalstamumas_metais_all = nusikalstamumas_all[nusikalstamumas_all['Mėnuo'] == 'Sausis - Gruodis']

nusikalstamumas_metais = nusikalstamumas_metais_all[['Metai', 'Savivaldybė', 'Nusikaltimai']]


nusikalstamumas_metais

,Metai,Savivaldybė,Nusikaltimai
8,2015,Respublika,68240
20,2015,Akmenės rajonas,570
32,2015,Joniškio rajonas,716
44,2015,Jurbarko rajonas,596
56,2015,Kaišiadorių rajonas,637
...,...,...,...
14069,2018,Druskininkai,238
14070,2018,Alytaus rajonas,404
14071,2018,Elektrėnai,349
14072,2018,Ignalinos rajonas,336


In [10]:
nusikalstamumas_metais['Nusikaltimai'] = (
    nusikalstamumas_metais['Nusikaltimai']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

nusikalstamumas_metais['Nusikaltimai'] = pd.to_numeric(
    nusikalstamumas_metais['Nusikaltimai'], errors='coerce'
)

# Patikrinam dtype
nusikalstamumas_metais['Nusikaltimai'].dtype

nusikalstamumas_metais

,Metai,Savivaldybė,Nusikaltimai
8,2015,Respublika,68240
20,2015,Akmenės rajonas,570
32,2015,Joniškio rajonas,716
44,2015,Jurbarko rajonas,596
56,2015,Kaišiadorių rajonas,637
...,...,...,...
14069,2018,Druskininkai,238
14070,2018,Alytaus rajonas,404
14071,2018,Elektrėnai,349
14072,2018,Ignalinos rajonas,336


In [11]:
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(nusikalstamumas_metais)

       Metai              Savivaldybė  Nusikaltimai
8       2015               Respublika         68240
20      2015          Akmenės rajonas           570
32      2015         Joniškio rajonas           716
44      2015         Jurbarko rajonas           596
56      2015      Kaišiadorių rajonas           637
68      2015   Kalvarijos savivaldybė           282
80      2015            Kauno miestas          9068
92      2015            Kauno rajonas          1688
104     2015  Kazlų Rūdos savivaldybė           190
116     2015           Kelmės rajonas           566
128     2015         Kėdainių rajonas          1066
140     2015        Klaipėdos miestas          4991
152     2015          Alytaus miestas          1206
164     2015        Klaipėdos rajonas           725
176     2015        Kretingos rajonas           368
188     2015         Kupiškio rajonas           437
200     2015          Lazdijų rajonas           430
212     2015          Marijampolės PK          1607
224     2015

In [12]:
# Sujungti Alytaus miestas ir Alytaus rajonas eilutes
alytus = nusikalstamumas_metais[nusikalstamumas_metais['Savivaldybė'].isin(['Alytaus miestas', 'Alytaus rajonas'])]

numeric_cols = ['Nusikaltimai']

# Apskaiciuoti skaitines reikšmes, pirmą ne-skaitinę reikšmę palikti kitiems stulpeliams
alytus_merged = alytus.groupby('Metai')[numeric_cols].sum().reset_index()
alytus_merged['Savivaldybė'] = 'Alytaus miestas ir rajonas'

# Pašalinti originalias Alytaus eilutes ir pridėti sujungtą
nusikalstamumas_metais = nusikalstamumas_metais[~nusikalstamumas_metais['Savivaldybė'].isin(['Alytaus miestas', 'Alytaus rajonas'])]
nusikalstamumas_metais = pd.concat([nusikalstamumas_metais, alytus_merged], ignore_index=True)

# Surūšiuoti
nusikalstamumas_metais = nusikalstamumas_metais.sort_values(['Metai', 'Savivaldybė']).reset_index(drop=True)

nusikalstamumas_metais

,Metai,Savivaldybė,Nusikaltimai
0,2004,Akmenės rajonas,470
1,2004,Alytaus miestas ir rajonas,1504
2,2004,Anykščių rajonas,704
3,2004,Birštonas,81
4,2004,Biržų rajonas,615
...,...,...,...
1315,2025,Šiaulių rajonas,434
1316,2025,Šilalės rajonas,150
1317,2025,Šilutės rajonas,478
1318,2025,Širvintų rajonas,143


In [13]:
len(nusikalstamumas_metais['Savivaldybė'].unique())

60

In [14]:
nusikalstamumas_metais['Savivaldybė'].unique()

<ArrowStringArray>
[           'Akmenės rajonas', 'Alytaus miestas ir rajonas',
           'Anykščių rajonas',                  'Birštonas',
              'Biržų rajonas',               'Druskininkai',
                 'Elektrėnai',          'Ignalinos rajonas',
            'Jonavos rajonas',           'Joniškio rajonas',
           'Jurbarko rajonas',        'Kaišiadorių rajonas',
     'Kalvarijos savivaldybė',              'Kauno miestas',
              'Kauno rajonas',    'Kazlų Rūdos savivaldybė',
             'Kelmės rajonas',          'Klaipėdos miestas',
          'Klaipėdos rajonas',          'Kretingos rajonas',
           'Kupiškio rajonas',           'Kėdainių rajonas',
            'Lazdijų rajonas',            'Marijampolės PK',
           'Mažeikių rajonas',             'Molėtų rajonas',
                    'Neringa',        'Pagėgių savivaldybė',
           'Pakruojo rajonas',                    'Palanga',
          'Panevėžio miestas',          'Panevėžio rajonas',
     

In [15]:
nusikalstamumas_metais = nusikalstamumas_metais[nusikalstamumas_metais['Savivaldybė'] != 'Respublika']

In [16]:
koordinates = {
    'Akmenės rajonas':              (56.2456, 22.7478),
    'Alytaus miestas ir rajonas':   (54.3963, 24.0457),
    'Anykščių rajonas':             (55.5167, 25.1000),
    'Birštonas':                    (54.6167, 24.0167),
    'Biržų rajonas':                (56.2000, 24.7500),
    'Druskininkai':                 (53.9833, 23.9833),
    'Elektrėnai':                   (54.7833, 24.6667),
    'Ignalinos rajonas':            (55.3500, 26.1667),
    'Jonavos rajonas':              (55.0667, 24.2833),
    'Joniškio rajonas':             (56.2333, 23.6167),
    'Jurbarko rajonas':             (55.0833, 22.7667),
    'Kaišiadorių rajonas':          (54.8667, 24.4500),
    'Kalvarijos savivaldybė':       (54.4167, 23.2167),
    'Kauno miestas':                (54.8985, 23.9036),
    'Kauno rajonas':                (54.9000, 23.8000),
    'Kazlų Rūdos savivaldybė':      (54.7500, 23.5000),
    'Kelmės rajonas':               (55.6333, 22.9333),
    'Klaipėdos miestas':            (55.7033, 21.1442),
    'Klaipėdos rajonas':            (55.6500, 21.3000),
    'Kretingos rajonas':            (55.8833, 21.2333),
    'Kupiškio rajonas':             (55.8333, 24.9667),
    'Kėdainių rajonas':             (55.2833, 23.9833),
    'Lazdijų rajonas':              (54.2333, 23.5167),
    'Marijampolės PK':              (54.5594, 23.3544),
    'Mažeikių rajonas':             (56.3167, 22.3333),
    'Molėtų rajonas':               (55.2333, 25.4167),
    'Neringa':                      (55.4667, 21.0833),
    'Pagėgių savivaldybė':          (55.1333, 21.9000),
    'Pakruojo rajonas':             (56.0667, 23.8667),
    'Palanga':                      (55.9167, 21.0667),
    'Panevėžio miestas':            (55.7333, 24.3667),
    'Panevėžio rajonas':            (55.7000, 24.3333),
    'Pasvalio rajonas':             (56.0600, 24.4033),
    'Plungės rajonas':              (55.9167, 21.8500),
    'Prienų rajonas':               (54.6333, 23.9500),
    'Radviliškio rajonas':          (55.8000, 23.5333),
    'Raseinių rajonas':             (55.3667, 23.1167),
    #'Respublika':                   (55.1694, 23.8813),  # Lietuvos centras
    'Rietavo savivaldybė':          (55.7333, 21.9167),
    'Rokiškio rajonas':             (55.9667, 25.5833),
    'Skuodo rajonas':               (56.2667, 21.5333),
    'Tauragės rajonas':             (55.2500, 22.2833),
    'Telšių rajonas':               (55.9833, 22.3500),
    'Trakų rajonas':                (54.6333, 24.9333),
    'Ukmergės rajonas':             (55.2500, 24.7667),
    'Utenos rajonas':               (55.5000, 25.6000),
    'Varėnos rajonas':              (54.2167, 24.5667),
    'Vilkaviškio rajonas':          (54.6500, 23.0333),
    'Vilniaus miestas':             (54.6872, 25.2797),
    'Vilniaus rajonas':             (54.8000, 25.5000),
    'Visaginas':                    (55.6000, 26.4333),
    'Zarasų rajonas':               (55.7333, 26.2500),
    'Šakių rajonas':                (54.9500, 23.0500),
    'Šalčininkų rajonas':           (54.3167, 25.3833),
    'Šiaulių miestas':              (55.9333, 23.3167),
    'Šiaulių rajonas':              (55.8667, 23.1833),
    'Šilalės rajonas':              (55.4833, 22.1833),
    'Šilutės rajonas':              (55.3500, 21.4667),
    'Širvintų rajonas':             (55.0500, 24.9500),
    'Švenčionių rajonas':           (55.1333, 26.0000),
}

nusikalstamumas_metais['lat'] = nusikalstamumas_metais['Savivaldybė'].map(lambda x: koordinates[x][0])
nusikalstamumas_metais['lon'] = nusikalstamumas_metais['Savivaldybė'].map(lambda x: koordinates[x][1])
nusikalstamumas_metais

,Metai,Savivaldybė,Nusikaltimai,lat,lon
0,2004,Akmenės rajonas,470,56.2456,22.7478
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457
2,2004,Anykščių rajonas,704,55.5167,25.1000
3,2004,Birštonas,81,54.6167,24.0167
4,2004,Biržų rajonas,615,56.2000,24.7500
...,...,...,...,...,...
1315,2025,Šiaulių rajonas,434,55.8667,23.1833
1316,2025,Šilalės rajonas,150,55.4833,22.1833
1317,2025,Šilutės rajonas,478,55.3500,21.4667
1318,2025,Širvintų rajonas,143,55.0500,24.9500


In [17]:
nusikalstamumas_metais.columns.tolist()

['Metai', 'Savivaldybė', 'Nusikaltimai', 'lat', 'lon']

# Pridedame kitas statistikas

In [18]:
from thefuzz import process

def match_savivaldybes(df_main, df_new, col_main='Savivaldybė', col_new='Savivaldybė'):
    """Automatiškai suranda geriausią pavadinimo atitikmenį"""
    unique_main = df_main[col_main].unique()
    unique_new = df_new[col_new].unique()

    matches = {}
    for name in unique_new:
        best_match, score = process.extractOne(name, unique_main)
        matches[name] = (best_match, score)

    return pd.DataFrame([
        {'Originalus': k, 'Surastas': v[0], 'Tikslumas': v[1]}
        for k, v in matches.items()
    ]).sort_values('Tikslumas')


In [19]:
def sujungimas(df, failas, stulpelio_pav):
    rod = pd.read_csv(f"statistikoa savivaldybem/{failas}")

    rod = rod.rename(
        columns={"Administracinė teritorija": "Savivaldybė"}
    )

    rod = rod.rename(
        columns={"Laikotarpis": "Metai"}
    )

    rod = rod[['Savivaldybė', 'Reikšmė', 'Metai']]

    rod['Savivaldybė'] = rod['Savivaldybė'].replace({
    'Alytaus m. sav.': 'Alytaus sav.',
    'Alytaus r. sav.': 'Alytaus sav.'
    })

    rod['Reikšmė'] = rod.groupby(['Savivaldybė', 'Metai'])['Reikšmė'].transform('mean')
    rod = rod.drop_duplicates(subset=['Savivaldybė', 'Metai'])

    match_df = match_savivaldybes(df, rod)
    print(match_df.to_string())

    manual_fixes = {
    'Vilniaus r. sav.': 'Vilniaus rajonas',
    'Kauno r. sav.':    'Kauno rajonas',
    "Klaipėdos r. sav.": 'Klaipėdos rajonas',
    "Panevėžio r. sav.": "Panevėžio rajonas",
    "Šiaulių r. sav.": "Šiaulių rajonas",

}

    # Sukurti pilną mapping
    mapping = {row['Originalus']: row['Surastas'] for _, row in match_df.iterrows()}
    mapping.update(manual_fixes)  # rankiniai pataisymai perrašo automatinius

    # Pritaikyti
    rod['Savivaldybė'] = rod['Savivaldybė'].map(mapping)

    print()
    print("---------PATIKTINTI AR LIKO NA------------")
    # Patikrinti ar liko NaN (nerasti atitikmenys)
    print(rod[rod['Savivaldybė'].isna()])
    print()

    print(rod[rod.isnull().any(axis=1)].to_string())
    rod = rod.dropna()

    df = df.merge(
    rod.rename(columns={'Reikšmė': stulpelio_pav}),
    on=['Savivaldybė', 'Metai'],
    how='left'
)
    print()
    print("---------SUJUNGTAS------------")
    print(df.shape)
    print(df.head(10))

    return df

## Gyventojų skaičius

In [20]:
df = sujungimas(nusikalstamumas_metais, 'Gyventoju_tankis_vienam_km2.csv', "Gyventoju_tankis")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10
3,2004,Birštonas,81,54.6167,24.0167,42.30
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20
...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10


## Bedarbių skaičius

In [21]:
df = sujungimas(df, 'bedarbiu_skaicius.csv', "Bedarbiu_sk")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0
...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0


# Imigrantų skaičius

In [22]:
df = sujungimas(df, 'imigrantu_skaicius.csv', "Imigrantu_sk")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0
...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0


# Medianinis gyventojų amžius

In [23]:
df = sujungimas(df, 'medianinis_gyventoju_amzius.csv', "Medianinis_amzius")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0
...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0


## Moterų skaičius tenkantis vienam tūkstančiui vyrų

In [24]:
df = sujungimas(df, 'moteru_skaicius_tenkantis_tukstanciui_vyru.csv', "Moteru")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0
...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0


## Nefinansinių įmonių apyvarta

In [25]:
df = sujungimas(df, 'nefinansiniu_imoniu_apyvarta.csv', "Apyvarta")
df

             Originalus                    Surastas  Tikslumas
4         Trakų r. sav.               Trakų rajonas         64
32        Biržų r. sav.               Biržų rajonas         64
30        Šakių r. sav.               Šakių rajonas         64
16        Kauno r. sav.               Kauno miestas         64
58       Zarasų r. sav.              Zarasų rajonas         67
40       Kelmės r. sav.              Kelmės rajonas         67
56       Utenos r. sav.              Utenos rajonas         67
55       Molėtų r. sav.              Molėtų rajonas         67
25       Skuodo r. sav.              Skuodo rajonas         67
52       Telšių r. sav.              Telšių rajonas         67
18       Prienų r. sav.              Prienų rajonas         67
44      Šiaulių r. sav.             Šiaulių miestas         69
38      Akmenės r. sav.             Akmenės rajonas         69
50      Plungės r. sav.             Plungės rajonas         69
26      Šilutės r. sav.             Šilutės rajonas    

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN


## Policijos pareigūnų skaičius

In [26]:
df = sujungimas(df, 'policijos_pareigunu_skaicius.csv', "Policija")
df

             Originalus                    Surastas  Tikslumas
4         Trakų r. sav.               Trakų rajonas         64
32        Biržų r. sav.               Biržų rajonas         64
30        Šakių r. sav.               Šakių rajonas         64
16        Kauno r. sav.               Kauno miestas         64
58       Zarasų r. sav.              Zarasų rajonas         67
40       Kelmės r. sav.              Kelmės rajonas         67
56       Utenos r. sav.              Utenos rajonas         67
55       Molėtų r. sav.              Molėtų rajonas         67
25       Skuodo r. sav.              Skuodo rajonas         67
52       Telšių r. sav.              Telšių rajonas         67
18       Prienų r. sav.              Prienų rajonas         67
44      Šiaulių r. sav.             Šiaulių miestas         69
38      Akmenės r. sav.             Akmenės rajonas         69
50      Plungės r. sav.             Plungės rajonas         69
26      Šilutės r. sav.             Šilutės rajonas    

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN


## Skurdo rizikos lygis

In [27]:
df = sujungimas(df, 'skurdo_rizikos_lygis_nuo_2010.csv', "Skurdas")
df

             Originalus                    Surastas  Tikslumas
4         Trakų r. sav.               Trakų rajonas         64
32        Biržų r. sav.               Biržų rajonas         64
30        Šakių r. sav.               Šakių rajonas         64
16        Kauno r. sav.               Kauno miestas         64
58       Zarasų r. sav.              Zarasų rajonas         67
40       Kelmės r. sav.              Kelmės rajonas         67
56       Utenos r. sav.              Utenos rajonas         67
55       Molėtų r. sav.              Molėtų rajonas         67
25       Skuodo r. sav.              Skuodo rajonas         67
52       Telšių r. sav.              Telšių rajonas         67
18       Prienų r. sav.              Prienų rajonas         67
44      Šiaulių r. sav.             Šiaulių miestas         69
38      Akmenės r. sav.             Akmenės rajonas         69
50      Plungės r. sav.             Plungės rajonas         69
26      Šilutės r. sav.             Šilutės rajonas    

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1


## Socialinę pašalpą gaunatys

In [28]:
df = sujungimas(df, 'socialine_pasalpa_gaunanciu_asmenu_skaicius.csv', "Pasalpa_gaunantys")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0


## Gyventojų skaičius

In [29]:
df = sujungimas(df, 'gyventoju_skaicius.csv', "gyventoju_skaicius")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0


## Vaikai nesimokantys mokykloje

In [30]:
vaikai = pd.read_csv('statistikoa savivaldybem/Vaikai_nesimokantys_mokykloje_nors_turetu_nuo_2009.csv')

vaikai['Laikotarpis'] = vaikai['Laikotarpis'].str.split('-').str[1]

vaikai.to_csv("statistikoa savivaldybem/Vaikai_nesimokantys_mokykloje_nors_turetu_nuo_2009_pakeista.csv", index=False)

df = sujungimas(df, 'Vaikai_nesimokantys_mokykloje_nors_turetu_nuo_2009_pakeista.csv', "Vaikai_nesimokantys")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius,Vaikai_nesimokantys
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0,NaN
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0,NaN
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0,NaN
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0,NaN
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0,115.0
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0,140.0
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0,361.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0,60.0


## Vidutinis uždarbis

In [31]:
uzdarbis = pd.read_csv('statistikoa savivaldybem/vidutinis_menesinis_darbo_uzmokestis_bruto_nuo_2011.csv')

uzdarbis['Laikotarpis'] = uzdarbis['Laikotarpis'].str[:4]

uzdarbis.to_csv("statistikoa savivaldybem/vidutinis_menesinis_darbo_uzmokestis_bruto_nuo_2011_pakeista.csv", index=False)
uzdarbis

,Laikotarpis,Rodiklis,Tipas,Administracinė teritorija,Lytis,Sektorius,Matavimo vienetai,Reikšmė
0,2011,Darbo užmokestis (mėnesinis),Bruto,Elektrėnų sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,613.2
1,2012,Darbo užmokestis (mėnesinis),Bruto,Elektrėnų sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,637.2
2,2013,Darbo užmokestis (mėnesinis),Bruto,Elektrėnų sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,690.9
3,2014,Darbo užmokestis (mėnesinis),Bruto,Elektrėnų sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,668.7
4,2015,Darbo užmokestis (mėnesinis),Bruto,Elektrėnų sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,708.9
...,...,...,...,...,...,...,...,...
910,2021,Darbo užmokestis (mėnesinis),Bruto,Zarasų r. sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,1122.7
911,2022,Darbo užmokestis (mėnesinis),Bruto,Zarasų r. sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,1270.6
912,2023,Darbo užmokestis (mėnesinis),Bruto,Zarasų r. sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,1379.0
913,2024,Darbo užmokestis (mėnesinis),Bruto,Zarasų r. sav.,Vyrai ir moterys,Šalies ūkis su individualiosiomis įmonėmis,EUR,1553.4


In [32]:
df = sujungimas(df, 'vidutinis_menesinis_darbo_uzmokestis_bruto_nuo_2011_pakeista.csv', "Uzmokestis")
df

              Originalus                    Surastas  Tikslumas
31         Šakių r. sav.               Šakių rajonas         64
16         Kauno r. sav.               Kauno miestas         64
4          Trakų r. sav.               Trakų rajonas         64
33         Biržų r. sav.               Biržų rajonas         64
59        Zarasų r. sav.              Zarasų rajonas         67
18        Prienų r. sav.              Prienų rajonas         67
53        Telšių r. sav.              Telšių rajonas         67
56        Molėtų r. sav.              Molėtų rajonas         67
25        Skuodo r. sav.              Skuodo rajonas         67
41        Kelmės r. sav.              Kelmės rajonas         67
57        Utenos r. sav.              Utenos rajonas         67
45       Šiaulių r. sav.             Šiaulių miestas         69
26       Šilutės r. sav.             Šilutės rajonas         69
51       Plungės r. sav.             Plungės rajonas         69
39       Akmenės r. sav.             Akm

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius,Vaikai_nesimokantys,Uzmokestis
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0,NaN,NaN
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0,NaN,NaN
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0,NaN,NaN
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0,NaN,NaN
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0,115.0,2033.3
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0,140.0,1879.9
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0,361.0,2022.0
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0,60.0,2153.4


## Alkoholio įperkamumas/vartojimas

In [33]:
alkoholis_iperkamumas = pd.read_csv('oficialios statistikos /alkoholio perkamoji galia.csv')
alkoholis_iperkamumas = alkoholis_iperkamumas[alkoholis_iperkamumas['Alkoholiniai gėrimai'].str.contains("Importinis alus skardinėse","litrai", na=False)]

In [34]:
alkoholis_iperkamumas

,Laikotarpis,Rodiklis,Alkoholiniai gėrimai,Matavimo vienetai,Reikšmė
13,2024,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,475.0
28,2023,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,437.0
43,2022,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,441.0
58,2021,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,436.0
73,2020,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,415.0
88,2019,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,370.0
103,2018,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,327.0
118,2017,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,306.0
133,2016,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,307.0
148,2015,Vidutinio mėnesinio neto darbo užmokesčio alko...,Importinis alus skardinėse,litrai,301.0


In [35]:
def priskirti_reiksme(pagrindinis_df, saltinis_df, naujas_stulpelis, metai_stulpelis='Metai', laikotarpis_stulpelis='Laikotarpis', reiksme_stulpelis='Reikšmė'):
    for _, eilute in saltinis_df.iterrows():
        laikotarpis = eilute[laikotarpis_stulpelis]
        reiksme = eilute[reiksme_stulpelis]
        pagrindinis_df.loc[pagrindinis_df[metai_stulpelis] == laikotarpis, naujas_stulpelis] = reiksme
    return pagrindinis_df

df = priskirti_reiksme(df,alkoholis_iperkamumas, 'alkoholio_iperkamumas')
df

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius,Vaikai_nesimokantys,Uzmokestis,alkoholio_iperkamumas
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0,NaN,NaN,NaN
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0,NaN,NaN,NaN
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0,NaN,NaN,NaN
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0,NaN,NaN,NaN
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0,115.0,2033.3,NaN
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0,140.0,1879.9,NaN
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0,361.0,2022.0,NaN
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0,60.0,2153.4,NaN


In [36]:
alkoholis_suvartojimas = pd.read_csv('oficialios statistikos /alkoholio suvartojimas vienam gyventojui.csv')
alkoholis_suvartojimas

,Laikotarpis,Rodiklis,Matavimo vienetai,Reikšmė
0,2024,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",8.8
1,2023,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",9.3
2,2022,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",9.5
3,2021,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",10.3
4,2020,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",9.7
5,2019,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",9.5
6,2018,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",9.6
7,2017,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",10.5
8,2016,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",11.6
9,2015,"Legalių alkoholinių gėrimų suvartojimas, tenka...","litrais, absoliutaus (100%) alkoholio",12.4


In [37]:
df = priskirti_reiksme(df,alkoholis_suvartojimas, 'alkoholio_suvartojimas')
df

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius,Vaikai_nesimokantys,Uzmokestis,alkoholio_iperkamumas,alkoholio_suvartojimas
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0,NaN,NaN,NaN,10.1
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0,NaN,NaN,NaN,10.1
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0,NaN,NaN,NaN,10.1
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0,NaN,NaN,NaN,10.1
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0,NaN,NaN,NaN,10.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0,115.0,2033.3,NaN,NaN
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0,140.0,1879.9,NaN,NaN
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0,361.0,2022.0,NaN,NaN
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0,60.0,2153.4,NaN,NaN


# Tabako suvartojimas

In [38]:
tabako_suvartojimas = pd.read_csv('oficialios statistikos /tabako suvartojimas.csv')
tabako_suvartojimas

,Laikotarpis,Rodiklis,Tabako gaminiai,Matavimo vienetai,Reikšmė
0,2024,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1009
1,2023,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1043
2,2022,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1062
3,2021,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1050
4,2020,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,957
5,2019,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,972
6,2018,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,988
7,2017,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1023
8,2016,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1094
9,2015,"Legalių tabako gaminių suvartojimas, tenkantis...",Tabako gaminiai,cigaretės,1100


In [39]:
df = priskirti_reiksme(df,tabako_suvartojimas, 'tabako_suvartojimas')
df

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius,Vaikai_nesimokantys,Uzmokestis,alkoholio_iperkamumas,alkoholio_suvartojimas,tabako_suvartojimas
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0,NaN,NaN,NaN,10.1,1154.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0,NaN,NaN,NaN,10.1,1154.0
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0,NaN,NaN,NaN,10.1,1154.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0,NaN,NaN,NaN,10.1,1154.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0,NaN,NaN,NaN,10.1,1154.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0,115.0,2033.3,NaN,NaN,NaN
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0,140.0,1879.9,NaN,NaN,NaN
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0,361.0,2022.0,NaN,NaN,NaN
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0,60.0,2153.4,NaN,NaN,NaN


In [40]:
df.to_csv("df.csv", index=False)